# v1 commissioning (read-only)

One section per implementation stage (S0–S9). Cells only call functions from `twobunch_deploy`; each stage saves its records under `data/` (git-ignored, copied back by hand), and the result goes in the lab-notebook commissioning page.

Nothing here writes to the machine: `tests/test_read_only.py` fails on any write call in the package or in these cells.

## S0: environment

Reads no PV. Run on the host in the ACR prod Python environment, from the repo root (Jupyter started inside tmux).

Pass:
- `import_errors` is empty;
- `code_sha256` equals the value CI printed for this PR's head commit (the `pixi run code-hash` step);
- `git_commit` is that commit;
- `epics_env` shows the host's Channel Access settings (recorded; no target value).

In [ ]:
from twobunch_deploy import session

env = session.check_env()
env

In [ ]:
session.save_record("S0", "check_env", env)

## S1: camera frames and pulse IDs

Reads one PV, `CAMR:LI20:107:RAW:ArrayData` (the DTOTR2 camera image as the driver outputs it, the same arrays the DAQ saves; `Image:ArrayData` until S2): subscribes for 30 s, then disconnects. Needs the camera on, and beam for real images. Saves the summary as JSON and, next to it (`.npz`), every frame's pulse ID and timestamps plus 10 raw frames.

Pass:
- `frames` > 0 and `latency_s.first` under about 1 s: the camera is publishing, and the frame timestamps agree with this host's clock;
- `lengths` is a single value, the camera ROI's `SizeX` × `SizeY` (278,640 = 324 × 860 for the ROI read on 10-30 at 15:26; S2 checks the geometry);
- `min` ≥ 0 and `max` ≤ 4095 (12-bit camera); `dtype` recorded;
- `pulse_id_steps` almost all 36. A 0 next to a 72, or a 35 next to a 37, is the IOC stamping a frame with a neighbouring pulse ID (8 times in 33,779 DAQ frames); any other step is a finding;
- `capture_fraction` close to 1: the callback should get every frame. Lower means frames were lost between the IOC and the notebook.

In [ ]:
from twobunch_deploy import session

summary = session.capture_test(seconds=30)
summary

## S2: frame geometry

Reads the DTOTR2 camera readbacks once (`pvs.CAMERA_SETTINGS`: ROI, orientation, exposure, gain, pixel size, the driver's reverse and output size, and where the camera array comes from and its size), then subscribes to the camera for 3 s. From each frame it cuts the deployed analysis's hand crop (`screen.HAND_CROP`, 300 × 450; at the training orientation sensor columns 396–579 and rows 270–719, padded with zeros past row 184 as the DAQ classes' crop does). The crop is taken in sensor pixels, whatever ROI the camera is set to, and oriented with the live orientation readbacks. Needs beam on DTOTR2: run the S1 section again first, with beam, then this one. Saves the summary as JSON and, next to it, the pulse IDs, the newest raw frame and every frame's hand crop (`.npz`) and the figure (`.png`).

Pass:
- no `TimeoutError`: every readback answers, so the PV names are right;
- no `ValueError`: the camera array comes from the driver (port `PS107`), isn't reversed, and is the camera ROI's size (no binning); the camera's ROI contains the hand crop; each frame is `SizeX` × `SizeY` pixels;
- `trained_shape` is [300, 450];
- `differs` lists the readbacks that aren't the training values. The four ROI readbacks may differ. Orientation, exposure (`AcquireTime_RBV`: 0.1 ms in training, 1 ms on 10-30), gain or resolution (`RESOLUTION`: 30.5 µm in training, 31.22 on 10-30) differing is a finding to understand before S3: the code follows the live orientation, but the analysis constants were set with the training values;
- the camera ROI didn't change during the 3 s: the readbacks are read once, before the capture, and a move that keeps its size wouldn't be caught. Ask the operator, or run the cell twice and compare `camera`;
- the figure's left panel shows the beam where the ops DTOTR2 display shows it, with the same handedness (compare with the display, or a screenshot of it);
- the beam lies inside the outlined hand crop.

In [ ]:
from twobunch_deploy import session

geometry = session.geometry_test(seconds=3)
geometry